## This script produces the arctic mean monthly AHT, OHU and IRF_aer

we keep various other variables in case they are useful

In [30]:
import numpy as np
import xarray as xr
import os
import glob
import matplotlib.pyplot as plt
import pandas as pd
import json
import cftime
from cftime import DatetimeNoLeap
from scipy import stats
from scipy.interpolate import interp1d
import dask
from tqdm import tqdm
from mpl_axes_aligner import align
from xmip.preprocessing import rename_cmip6
%matplotlib inline


from utils import QACC_utils
#from utils.config import model, min_lat_arctic, max_lat_arctic, min_lat_tropics, max_lat_tropics, min_year_late_cent, max_year_late_cent

##### SETTINGS
model = 'UKESM1-0-LL'
#min_lat_arctic, max_lat_arctic = 66, 90
#min_lat_tropics, max_lat_tropics = -23, 23


regions = ['Arctic', 'Tropics']
lat_band_dict = {'Arctic':[66, 90],
                 'Tropics':[-23, 23]}

scenarios = ['ssp245', 'ARISE', 'ssp245_baseline']

experiment_dict = {'ssp245':'ssp245', 
                   'ARISE':'ARISE',
                   'ssp245_baseline':'ssp245'}

time_slice_dict = {'ssp245':['2050', '2070'], 
                   'ARISE':['2050', '2070'],
                   'ssp245_baseline':['2015', '2030']}

## Part 1
presave a set of needed variables, as arctic means

In [27]:
### define set of required variables, keep more than are needed here

rad_flux_vars_surface = ['rsds', 'rsus', 'rlds', 'rlus', 'rsdscs', 'rldscs']
# all radiative vars at surface

rad_flux_vars_TOA = ['rsdt', 'rsut', 'rlut', 'rlutcs', 'rsutcs', 'rsutaf', 'rsutcsaf']
# all radiative vars at top-of-atmosphere

heat_flux_vars_surface = ['hfls', 'hfss'] 
# surface upward latent heat flux and surface upward sensible heat flux

PE_vars = ['evspsbl', 'pr']
# precip and evap vars needed to distinguish sensible and latent heat

column_stored_energy_vars = ['ta', 'hus', 'hur']

additional = ['huss'] # might need the surface humidity for q feedback

# see Donohue et al 2020 JCLIM, equation A2 https://faculty.washington.edu/karmour/papers/Donohoe_etal_JCLIM2020.pdf

AHT_vars = ['tas'] + rad_flux_vars_surface + rad_flux_vars_TOA + heat_flux_vars_surface + PE_vars + additional # + column_stored_energy_vars

In [28]:
def get_monthlyQ(ds):

    ## accounting for storage makes no difference : delta_S is < 0.001 W/m2
    
    
    # Constants
    c_p = 1004        # J/(kg·K), specific heat of dry air
    g = 9.81         # m/s², gravity
    L = 2.5e6       # J/kg, latent heat of vaporization
    seconds_per_month = 2.592e6 
    S = (1/g) * (c_p*ds_ssp245.ta.sum('plev') + L*ds_ssp245.hus.sum('plev')) * (1/seconds_per_month)
    delta_S = S - S.roll(month=1)

    ds['OHU'] = ds.rsds - ds.rsus + ds.rlds - ds.rlus - ds.hfls - ds.hfss
    ds['AHT'] = ds.rsut + ds.rlut - ds.rsdt + delta_S + ds['OHU']
    
    ### now split into moist and dry components. Done following Hahn et al. 2021
    ### https://www.frontiersin.org/articles/10.3389/feart.2021.710036/full
    ### we take moist energy transport as (P-E)*(latent heat of vaporisation).
    ### this neglects the latent heat of fusion for solid precip. 
    ### the dry component is then calculated as the difference between Q and moist

    L = 2.501*(10**6)# latent heat of vaporisation, assumed constant at 2.501 × 10^6 J kg−1
    ### OLD VERSION, which used evspsbl:  ###  ds['Qmoist'] = (ds.pr - ds.evspsbl)*L*unit_conversions[units]  
    ## new, uses hfls instead:
    ds['AHTmoist'] = ds.pr*L - ds.hfls 
    ds['AHTdry'] = ds['AHT'] - ds['AHTmoist']

    
    return ds


def add_extra_variables(ds, scen, min_lat):
    ## adds F, albedo, and CRE, and saves:
    # add IRF, allsky and clearsky
    ds['IRF_aer'] = ds['rsutaf'] - ds['rsut']  # sign convention here is that more SW to space is a negative forcing
    ds['IRF_aer_cs'] = ds['rsutcsaf'] - ds['rsutcs']

    # add albedo, defined as ratio of reflected up to incident down SW radiation:
    ds['albedo'] = ds['rsus']/ds['rsds']

    # add CRE (cloud radiative effect))
    ds['CRE'] = ((ds['rsutcs']-ds['rsut'])+(ds['rlutcs']-ds['rlut']))
    
    ds.to_netcdf('intermediate_outputs/fin/{s}_{l}_{m}.nc'.format(
        s=scen, l=str(min_lat), m=model))
    return ds

In [31]:
## v2, do all in one go
### preproces, a couple minutes:


for scen in scenarios:
    print(scen)
    for region in regions:
        
        print(region)
    
        # single level vars first:
        ds_sl = QACC_utils.get_all_vars_arctic_monthly(vars=AHT_vars,
                                                       model=model,
                                                       scenario=experiment_dict[scen],
                                                       min_lat=lat_band_dict[region][0],
                                                       max_lat=lat_band_dict[region][1],
                                                       min_year=time_slice_dict[scen][0], 
                                                       max_year=time_slice_dict[scen][1])
        
        ds_sl.to_netcdf('intermediate_outputs/init/{s}_{l}_{m}_{r}_singlelev.nc'.format(
            s=scen, l=str(lat_band_dict[region][0]), m=model, r=region))
    
    
        # repeat for the vars on levels
        ds_al = QACC_utils.get_all_vars_arctic_monthly(vars=column_stored_energy_vars, model=model,
                                                       scenario=experiment_dict[scen],
                                                       min_lat=lat_band_dict[region][0],
                                                       max_lat=lat_band_dict[region][1],
                                                       min_year=time_slice_dict[scen][0], 
                                                       max_year=time_slice_dict[scen][1])
        
        ds_al.to_netcdf('intermediate_outputs/init/{s}_{l}_{m}_{r}_alllevs.nc'.format(
            s=scen, l=str(lat_band_dict[region][0]), m=model, r=region))

        ds = xr.merge([ds_sl, ds_al], compat='override') # override necessary because of different number of ensemble members, which we saved as a var
        print('made combined ds')

        # add heat transport terms
        ds = get_monthlyQ(ds)
        print('added Q')

        # add final calculated vars (albedo, CRE, IRF), and save
        ds = add_extra_variables(ds, scen, lat_band_dict[region][0])
        print('done')
        

ssp245
Arctic


  0%|          | 0/19 [00:00<?, ?it/s]

tas


  5%|▌         | 1/19 [00:01<00:27,  1.55s/it]

rsds


...

hus


 67%|██████▋   | 2/3 [00:01<00:00,  1.19it/s]

hur


100%|██████████| 3/3 [00:02<00:00,  1.21it/s]


made combined ds
added Q
done
